In [ ]:
# 01 Data Understanding

## 1. Imports and paths

## 2. Load source tables

## 3. Dataset overview

## 4. Table-by-table inspection

## 5. Primary key and foreign key checks

## 6. Missing values and duplicates

## 7. Relationship map

## 8. Initial business questions

## 9. Decisions and data limitations

In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

In [ ]:
files = {
    "transactions": "transaction_data.csv",
    "products": "product.csv",
    "households": "hh_demographic.csv",
    "campaign_desc": "campaign_desc.csv",
    "campaign_table": "campaign_table.csv",
    "coupon": "coupon.csv",
    "coupon_redemptions": "coupon_redempt.csv",
}

In [ ]:
tables = {}

for name, file in files.items():
    path = RAW_DIR / file
    tables[name] = pd.read_csv(path)

    print(f"{name}: {tables[name].shape}")

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*60}")
    print(name.upper())
    print(f"Shape: {df.shape}")
    print("\nColumns and dtypes:")
    print(df.dtypes)
    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
causal_sample = pd.read_csv(
    RAW_DIR / "causal_data.csv",
    nrows=5
)

print(causal_sample.dtypes)
display(causal_sample)

In [ ]:
transactions = tables["transactions"]

print("Rows:", len(transactions))
print("Unique baskets:", transactions["BASKET_ID"].nunique())
print("Unique households:", transactions["household_key"].nunique())
print("Unique products:", transactions["PRODUCT_ID"].nunique())
print("Unique stores:", transactions["STORE_ID"].nunique())

In [ ]:
transactions.groupby("BASKET_ID").size().describe()

In [ ]:
duplicate_basket_product = transactions.duplicated(
    subset=["BASKET_ID", "PRODUCT_ID"]
).sum()

print("Duplicate basket-product combinations:", duplicate_basket_product)

In [ ]:
products = tables["products"]

print("Rows:", len(products))
print("Unique PRODUCT_ID:", products["PRODUCT_ID"].nunique())

duplicate_product_ids = products.duplicated(
    subset=["PRODUCT_ID"]
).sum()

print("Duplicate PRODUCT_ID rows:", duplicate_product_ids)

In [ ]:
transaction_product_ids = set(transactions["PRODUCT_ID"])
product_master_ids = set(products["PRODUCT_ID"])

missing_product_ids = transaction_product_ids - product_master_ids

print("Transaction products missing from product table:", len(missing_product_ids))

In [ ]:
households = tables["households"]

transaction_households = set(transactions["household_key"])
demographic_households = set(households["household_key"])

matched_households = transaction_households & demographic_households
missing_demographics = transaction_households - demographic_households

print("Transaction households:", len(transaction_households))
print("Households with demographics:", len(matched_households))
print("Households without demographics:", len(missing_demographics))
print(
    "Demographic coverage:",
    round(len(matched_households) / len(transaction_households) * 100, 2),
    "%"
)

In [ ]:
print("Rows:", len(households))
print("Unique household_key:", households["household_key"].nunique())

duplicate_households = households.duplicated(
    subset=["household_key"]
).sum()

print("Duplicate household_key rows:", duplicate_households)

In [ ]:
campaign_desc = tables["campaign_desc"]

print("Rows:", len(campaign_desc))
print("Unique CAMPAIGN values:", campaign_desc["CAMPAIGN"].nunique())

duplicate_campaigns = campaign_desc.duplicated(
    subset=["CAMPAIGN"]
).sum()

print("Duplicate CAMPAIGN rows:", duplicate_campaigns)

In [ ]:
campaign_table = tables["campaign_table"]

campaigns_per_household = (
    campaign_table.groupby("household_key")["CAMPAIGN"]
    .nunique()
)

print("Unique households in campaign table:", campaign_table["household_key"].nunique())
print("Max campaigns for one household:", campaigns_per_household.max())
print("Average campaigns per household:", round(campaigns_per_household.mean(), 2))

In [ ]:
campaign_check = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION"]],
    on="CAMPAIGN",
    how="left",
    suffixes=("_table", "_desc")
)

mismatches = campaign_check[
    campaign_check["DESCRIPTION_table"] != campaign_check["DESCRIPTION_desc"]
]

print("Rows checked:", len(campaign_check))
print("Description mismatches:", len(mismatches))

In [ ]:
coupon = tables["coupon"]

products_per_coupon = (
    coupon.groupby("COUPON_UPC")["PRODUCT_ID"]
    .nunique()
)

coupons_per_product = (
    coupon.groupby("PRODUCT_ID")["COUPON_UPC"]
    .nunique()
)

print("Unique coupons:", coupon["COUPON_UPC"].nunique())
print("Max products for one coupon:", products_per_coupon.max())
print("Average products per coupon:", round(products_per_coupon.mean(), 2))

print("Unique products in coupon table:", coupon["PRODUCT_ID"].nunique())
print("Max coupons for one product:", coupons_per_product.max())
print("Average coupons per product:", round(coupons_per_product.mean(), 2))

In [ ]:
coupon_redemptions = tables["coupon_redemptions"]

print("Rows:", len(coupon_redemptions))

duplicate_redemptions = coupon_redemptions.duplicated(
    subset=["household_key", "DAY", "COUPON_UPC", "CAMPAIGN"]
).sum()

print("Exact duplicate redemption rows:", duplicate_redemptions)

print(
    "Unique households redeeming:",
    coupon_redemptions["household_key"].nunique()
)

print(
    "Unique coupons redeemed:",
    coupon_redemptions["COUPON_UPC"].nunique()
)

print(
    "Unique campaigns with redemptions:",
    coupon_redemptions["CAMPAIGN"].nunique()
)

In [ ]:
import pandas as pd
import numpy as np

causal_path = RAW_DIR / "causal_data.csv"

total_rows = 0
duplicate_keys = 0
sort_violations = 0
previous_last = None

for chunk in pd.read_csv(
    causal_path,
    usecols=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    chunksize=250_000
):
    p = chunk["PRODUCT_ID"].to_numpy()
    s = chunk["STORE_ID"].to_numpy()
    w = chunk["WEEK_NO"].to_numpy()

    total_rows += len(chunk)

    # Duplicate adjacent keys within the chunk
    duplicates = (
        (p[1:] == p[:-1]) &
        (s[1:] == s[:-1]) &
        (w[1:] == w[:-1])
    )
    duplicate_keys += duplicates.sum()

    # Check lexicographic sorting within the chunk
    violations = (
        (p[1:] < p[:-1]) |
        (
            (p[1:] == p[:-1]) &
            (s[1:] < s[:-1])
        ) |
        (
            (p[1:] == p[:-1]) &
            (s[1:] == s[:-1]) &
            (w[1:] < w[:-1])
        )
    )
    sort_violations += violations.sum()

    # Compare first row of this chunk with last row of previous chunk
    if previous_last is not None:
        current_first = (p[0], s[0], w[0])

        if current_first == previous_last:
            duplicate_keys += 1

        if current_first < previous_last:
            sort_violations += 1

    previous_last = (p[-1], s[-1], w[-1])

print("Total rows:", total_rows)
print("Duplicate adjacent keys:", duplicate_keys)
print("Sort violations:", sort_violations)

In [ ]:
causal_duplicates = []

previous_row = None

for chunk in pd.read_csv(
    causal_path,
    chunksize=250_000
):
    for row in chunk.itertuples(index=False):
        key = (row.PRODUCT_ID, row.STORE_ID, row.WEEK_NO)

        if previous_row is not None:
            previous_key = (
                previous_row.PRODUCT_ID,
                previous_row.STORE_ID,
                previous_row.WEEK_NO
            )

            if key == previous_key:
                causal_duplicates.append({
                    "PRODUCT_ID": row.PRODUCT_ID,
                    "STORE_ID": row.STORE_ID,
                    "WEEK_NO": row.WEEK_NO,
                    "display_prev": previous_row.display,
                    "display_current": row.display,
                    "mailer_prev": previous_row.mailer,
                    "mailer_current": row.mailer
                })

                if len(causal_duplicates) >= 20:
                    break

        previous_row = row

    if len(causal_duplicates) >= 20:
        break

pd.DataFrame(causal_duplicates)

In [ ]:
current_key = None
current_states = set()

duplicate_full_rows = 0
multi_state_groups = 0
total_groups = 0

for chunk in pd.read_csv(
    causal_path,
    chunksize=250_000
):
    for row in chunk.itertuples(index=False):

        key = (
            row.PRODUCT_ID,
            row.STORE_ID,
            row.WEEK_NO
        )

        state = (
            row.display,
            row.mailer
        )

        if current_key is None:
            current_key = key

        if key != current_key:

            total_groups += 1

            if len(current_states) > 1:
                multi_state_groups += 1

            current_key = key
            current_states = set()

        if state in current_states:
            duplicate_full_rows += 1

        current_states.add(state)

# Final group
if current_key is not None:
    total_groups += 1

    if len(current_states) > 1:
        multi_state_groups += 1

print("Product-store-week groups:", total_groups)
print("Groups with multiple promotion states:", multi_state_groups)
print("Exact duplicate full rows:", duplicate_full_rows)

In [ ]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    result = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    })
    
    display(result[result["missing_count"] > 0])

In [ ]:
for name, df in tables.items():
    duplicate_rows = df.duplicated().sum()
    print(f"{name}: {duplicate_rows} exact duplicate rows")

In [ ]:
coupon_duplicate_counts = (
    coupon.groupby(
        ["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"]
    )
    .size()
    .reset_index(name="row_count")
)

duplicate_combinations = coupon_duplicate_counts[
    coupon_duplicate_counts["row_count"] > 1
]

print("Duplicate combinations:", len(duplicate_combinations))
print("Maximum repetitions of one combination:", duplicate_combinations["row_count"].max())

display(
    duplicate_combinations
    .sort_values("row_count", ascending=False)
    .head(10)
)

In [ ]:
numeric_cols = [
    "DAY",
    "WEEK_NO",
    "QUANTITY",
    "SALES_VALUE",
    "RETAIL_DISC",
    "COUPON_DISC",
    "COUPON_MATCH_DISC",
    "TRANS_TIME"
]

transactions[numeric_cols].describe().T

In [ ]:
largest_quantity_rows = (
    transactions
    .nlargest(20, "QUANTITY")
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC",
                "SUB_COMMODITY_DESC",
                "CURR_SIZE_OF_PRODUCT"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

display(
    largest_quantity_rows[
        [
            "BASKET_ID",
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "DEPARTMENT",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC",
            "CURR_SIZE_OF_PRODUCT"
        ]
    ]
)

In [ ]:
high_quantity = transactions[
    transactions["QUANTITY"] > 100
]

print("Rows with QUANTITY > 100:", len(high_quantity))
print(
    "Percentage of transaction rows:",
    round(len(high_quantity) / len(transactions) * 100, 3),
    "%"
)

high_quantity_by_department = (
    high_quantity
    .merge(
        products[["PRODUCT_ID", "DEPARTMENT"]],
        on="PRODUCT_ID",
        how="left"
    )
    ["DEPARTMENT"]
    .value_counts()
)

display(high_quantity_by_department.head(15))

In [ ]:
largest_sales_rows = (
    transactions
    .nlargest(20, "SALES_VALUE")
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC",
                "SUB_COMMODITY_DESC",
                "CURR_SIZE_OF_PRODUCT"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

display(
    largest_sales_rows[
        [
            "BASKET_ID",
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "DEPARTMENT",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC",
            "CURR_SIZE_OF_PRODUCT"
        ]
    ]
)

In [ ]:
positive_retail_disc = (
    transactions[
        transactions["RETAIL_DISC"] > 0
    ]
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC",
                "SUB_COMMODITY_DESC"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

print(
    "Rows with positive RETAIL_DISC:",
    len(positive_retail_disc)
)

display(
    positive_retail_disc[
        [
            "BASKET_ID",
            "PRODUCT_ID",
            "SALES_VALUE",
            "RETAIL_DISC",
            "DEPARTMENT",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC"
        ]
    ]
    .sort_values("RETAIL_DISC", ascending=False)
    .head(20)
)

In [ ]:
real_positive_disc = transactions[
    transactions["RETAIL_DISC"] > 0.001
]

print("Rows with RETAIL_DISC > 0.001:", len(real_positive_disc))

print("\nPositive discount values:")
print(
    real_positive_disc["RETAIL_DISC"]
    .value_counts()
    .sort_index(ascending=False)
)

In [ ]:
print("COUPON_DISC > 0:",
      (transactions["COUPON_DISC"] > 0).sum())

print("COUPON_MATCH_DISC > 0:",
      (transactions["COUPON_MATCH_DISC"] > 0).sum())

print("Rows with COUPON_DISC < 0:",
      (transactions["COUPON_DISC"] < 0).sum())

print("Rows with COUPON_MATCH_DISC < 0:",
      (transactions["COUPON_MATCH_DISC"] < 0).sum())

In [ ]:
match_without_coupon = transactions[
    (transactions["COUPON_MATCH_DISC"] < 0) &
    (transactions["COUPON_DISC"] == 0)
]

print(
    "Rows with coupon match but no coupon discount:",
    len(match_without_coupon)
)

In [ ]:
trans_time = transactions["TRANS_TIME"]

hours = trans_time // 100
minutes = trans_time % 100

invalid_time = transactions[
    (hours > 23) |
    (minutes > 59)
]

print("Invalid TRANS_TIME rows:", len(invalid_time))
print("Rows with TRANS_TIME = 0:", (trans_time == 0).sum())

In [ ]:
midnight_rows = transactions[
    transactions["TRANS_TIME"] == 0
]

print("Midnight rows:", len(midnight_rows))
print("Unique baskets:", midnight_rows["BASKET_ID"].nunique())
print("Unique households:", midnight_rows["household_key"].nunique())
print("Unique stores:", midnight_rows["STORE_ID"].nunique())
print("Day range:", midnight_rows["DAY"].min(), "to", midnight_rows["DAY"].max())

In [ ]:
midnight_products = (
    midnight_rows
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

display(
    midnight_products["DEPARTMENT"]
    .value_counts()
    .head(10)
)

In [ ]:
print("Rows with SALES_VALUE < 0:",
      (transactions["SALES_VALUE"] < 0).sum())

print("Rows with SALES_VALUE = 0:",
      (transactions["SALES_VALUE"] == 0).sum())

In [ ]:
zero_sales = (
    transactions[transactions["SALES_VALUE"] == 0]
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC",
                "SUB_COMMODITY_DESC"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

print("Unique baskets with zero sales:", zero_sales["BASKET_ID"].nunique())

display(
    zero_sales["DEPARTMENT"]
    .value_counts()
    .head(10)
)

In [ ]:
print("Unique baskets with zero sales:", zero_sales["BASKET_ID"].nunique())

In [ ]:
blank_zero_sales = zero_sales[
    zero_sales["DEPARTMENT"].fillna("").str.strip() == ""
]

print("Blank-department zero-sales rows:", len(blank_zero_sales))

display(
    blank_zero_sales[
        [
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "RETAIL_DISC",
            "COUPON_DISC",
            "COUPON_MATCH_DISC",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC"
        ]
    ].head(20)
)

In [ ]:
products[
    products["PRODUCT_ID"].isin([5978648, 5978656])
]

In [ ]:
blank_zero_sales["PRODUCT_ID"].value_counts().head(10)

In [ ]:
zero_qty = transactions[transactions["QUANTITY"] == 0]

print("Total zero-quantity rows:", len(zero_qty))

print(
    "Zero quantity with positive sales:",
    (zero_qty["SALES_VALUE"] > 0).sum()
)

print(
    "Zero quantity with zero sales:",
    (zero_qty["SALES_VALUE"] == 0).sum()
)

zero_qty_products = zero_qty.merge(
    products[
        ["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]
    ],
    on="PRODUCT_ID",
    how="left"
)

display(
    zero_qty_products["DEPARTMENT"]
    .value_counts(dropna=False)
    .head(15)
)

In [ ]:
zero_qty_positive_sales = zero_qty_products[
    zero_qty_products["SALES_VALUE"] > 0
]

print(
    "Rows:",
    len(zero_qty_positive_sales)
)

display(
    zero_qty_positive_sales[
        [
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "DEPARTMENT",
            "COMMODITY_DESC"
        ]
    ].sort_values(
        "SALES_VALUE",
        ascending=False
    ).head(20)
)

In [ ]:
produce_high_qty = (
    transactions
    .merge(
        products[
            [
                "PRODUCT_ID",
                "DEPARTMENT",
                "COMMODITY_DESC",
                "SUB_COMMODITY_DESC",
                "CURR_SIZE_OF_PRODUCT"
            ]
        ],
        on="PRODUCT_ID",
        how="left"
    )
)

display(
    produce_high_qty[
        (produce_high_qty["DEPARTMENT"] == "PRODUCE") &
        (produce_high_qty["QUANTITY"] > 100)
    ][
        [
            "PRODUCT_ID",
            "QUANTITY",
            "SALES_VALUE",
            "RETAIL_DISC",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC",
            "CURR_SIZE_OF_PRODUCT"
        ]
    ]
)

In [ ]:
print("Unique departments:", products["DEPARTMENT"].nunique())
print("Unique brand values:", products["BRAND"].nunique())

print("\nDepartment distribution:")
display(products["DEPARTMENT"].value_counts(dropna=False))

print("\nBrand distribution:")
display(products["BRAND"].value_counts(dropna=False))

In [ ]:
category_cols = [
    "DEPARTMENT",
    "COMMODITY_DESC",
    "SUB_COMMODITY_DESC",
    "CURR_SIZE_OF_PRODUCT"
]

for col in category_cols:
    values = products[col].fillna("").astype(str).str.strip()

    print(f"\n{col}")
    print("Blank values:", values.eq("").sum())

    print("Most common values:")
    display(values.value_counts().head(10))

In [ ]:
blank_categories = products[
    products["DEPARTMENT"].fillna("").str.strip().eq("") &
    products["COMMODITY_DESC"].fillna("").str.strip().eq("") &
    products["SUB_COMMODITY_DESC"].fillna("").str.strip().eq("")
]

print("Products with all three categories blank:", len(blank_categories))

print(
    "Products appearing in transactions:",
    blank_categories["PRODUCT_ID"]
    .isin(transactions["PRODUCT_ID"])
    .sum()
)

display(blank_categories)

In [ ]:
weeks_per_day = (
    transactions
    .groupby("DAY")["WEEK_NO"]
    .nunique()
)

print("Unique days:", transactions["DAY"].nunique())

print(
    "Days mapped to multiple weeks:",
    (weeks_per_day > 1).sum()
)

print("\nFirst 15 day-week combinations:")

display(
    transactions[["DAY", "WEEK_NO"]]
    .drop_duplicates()
    .sort_values("DAY")
    .head(15)
)

In [ ]:
week_summary = (
    transactions.groupby("WEEK_NO")
    .agg(
        start_day=("DAY", "min"),
        end_day=("DAY", "max"),
        unique_days=("DAY", "nunique")
    )
    .reset_index()
)

display(week_summary.head(5))
display(week_summary.tail(5))

print(
    "Weeks with fewer than 7 days:",
    (week_summary["unique_days"] < 7).sum()
)

In [ ]:
campaign_reach = (
    campaign_table
    .groupby("CAMPAIGN")["household_key"]
    .nunique()
    .reset_index(name="households_targeted")
)

campaign_reach["reach_pct"] = (
    campaign_reach["households_targeted"]
    / transactions["household_key"].nunique()
    * 100
).round(2)

print(
    "Total unique households targeted:",
    campaign_table["household_key"].nunique()
)

print("\nCampaign reach summary:")
display(campaign_reach["households_targeted"].describe())

print("\nTop 5 campaigns by reach:")
display(
    campaign_reach
    .sort_values("households_targeted", ascending=False)
    .head(5)
)

In [ ]:
redemptions = tables["coupon_redemptions"]

targeted_pairs = campaign_table[
    ["household_key", "CAMPAIGN"]
].drop_duplicates()

redemption_check = redemptions.merge(
    targeted_pairs.assign(was_targeted=True),
    on=["household_key", "CAMPAIGN"],
    how="left"
)

print("Total redemption records:", len(redemption_check))

print(
    "Redemptions linked to a targeted household:",
    redemption_check["was_targeted"].eq(True).sum()
)

print(
    "Redemptions without matching campaign targeting:",
    redemption_check["was_targeted"].isna().sum()
)

print(
    "Unique households redeeming coupons:",
    redemptions["household_key"].nunique()
)

In [ ]:
redeemers_by_campaign = (
    redemptions
    .groupby("CAMPAIGN")["household_key"]
    .nunique()
    .reset_index(name="households_redeemed")
)

campaign_performance = campaign_reach.merge(
    redeemers_by_campaign,
    on="CAMPAIGN",
    how="left"
)

campaign_performance["households_redeemed"] = (
    campaign_performance["households_redeemed"]
    .fillna(0)
    .astype(int)
)

campaign_performance["redemption_rate_pct"] = (
    campaign_performance["households_redeemed"]
    / campaign_performance["households_targeted"]
    * 100
).round(2)

display(
    campaign_performance[
        [
            "CAMPAIGN",
            "households_targeted",
            "households_redeemed",
            "redemption_rate_pct"
        ]
    ]
    .sort_values("redemption_rate_pct", ascending=False)
    .head(10)
)

print(
    "Campaigns with redemption rate above 100%:",
    (campaign_performance["redemption_rate_pct"] > 100).sum()
)

In [ ]:
print(
    "Campaigns with redemption rate above 100%:",
    (campaign_performance["redemption_rate_pct"] > 100).sum()
)

In [ ]:
# Unique coupon-campaign combinations
valid_coupon_campaigns = coupon[
    ["COUPON_UPC", "CAMPAIGN"]
].drop_duplicates()

# Check redeemed coupons against their campaigns
coupon_validation = redemptions.merge(
    valid_coupon_campaigns,
    on=["COUPON_UPC", "CAMPAIGN"],
    how="left",
    indicator=True
)

print("Total redemptions:", len(coupon_validation))

print(
    "Valid coupon-campaign matches:",
    (coupon_validation["_merge"] == "both").sum()
)

print(
    "Unmatched coupon-campaign combinations:",
    (coupon_validation["_merge"] == "left_only").sum()
)

## Final Data Understanding Summary

### Completed checks
- Source table structures and relationships
- Transaction, product and household key validation
- Campaign and coupon relationship checks
- Nulls, blanks and duplicate records
- Quantity, sales and discount behaviour
- Transaction time validation
- Day and week consistency
- Campaign reach and coupon redemption analysis

### Main findings
1. Demographics cover only 32.04% of transacting households.
2. The coupon source contains 5,164 exact duplicate rows.
3. Fuel and miscellaneous transactions use unusual quantity scales.
4. Some positive-sales records have zero quantity.
5. The causal table contains multiple promotion states for some product-store-week combinations.
6. Fifteen products have no category classification.
7. The first and last reporting weeks are incomplete.
8. Coupon redemption records passed the household-campaign and coupon-campaign relationship checks.

### Modelling recommendations
- Preserve all original source files.
- Load the data into PostgreSQL.
- Clean and validate records using dbt staging models.
- Build analytical fact, dimension and bridge tables.
- Keep campaign and promotion relationships at appropriate grains.
- Prevent duplicate joins from inflating sales KPIs.
- Clearly document reporting exclusions, limitations and assumptions.

### Next Phase
PostgreSQL data loading and dbt transformation modelling.